# Tools and Agents (open source): Tagging and Extraction

### Outline
- Tagging: sentiment + language of a short text
- Extraction: structured people/ages out of free text
- Doing it for real: tag and extract from a live Wikipedia article, chunked

Open-source recode of `03-Functions-Tools-and-Agents-with-LangChain/L4-tagging-and-extraction-student.ipynb`.
The original chains `convert_pydantic_to_openai_function(...)`, `model.bind(functions=..., function_call={...})`
and `JsonOutputFunctionsParser()` / `JsonKeyOutputFunctionsParser(key_name=...)` by hand. Modern
LangChain collapses all of that into one call: `model.with_structured_output(PydanticModel)`
returns a Runnable that outputs a validated instance of the model directly.

`with_structured_output` defaults to `method="json_schema"` (Ollama's native structured-output
mode), but that came back as unparsed prose against this project's cloud model/account - pinning
`method="function_calling"` (same tool-calling machinery as L1/L3) is what actually works
reliably here.

The "doing it for real" section swaps the original's `WebBaseLoader` fetch of a blog post for a
Wikipedia article via the `wikipedia` package (same source L5/L6 use for their
`search_wikipedia` tool).

## Setup

This notebook is the open-source / Ollama-cloud recode of the matching lesson in
[`openai_agentic_ai_course`](../../openai_agentic_ai_course/), reusing the shared
`common.py` / `tracing.py` helpers already built for
[`tools_and_agent`](../../tools_and_agent/) rather than duplicating them here.

- **Model**: `ChatOllama`, pointed at the Ollama cloud endpoint configured in the
  repo-root `.env` (`OLLAMA_MODEL` / `OLLAMA_BASE_URL` / `OLLAMA_API_KEY`).
- **Tracing**: every `.invoke()` / `.batch()` / `.stream()` call below passes
  `config=traced("run name")`, which attaches a Langfuse callback - open the
  Langfuse dashboard and filter by trace name to see this notebook's calls.
- **Kernel**: run this with the repo's `.venv` (`Python 3 (ipykernel)`) - it already
  has everything in [`requirements.txt`](../../requirements.txt) installed.

In [ ]:
import sys
from pathlib import Path

# common.py / tracing.py live in tools_and_agent/, not here - add it to sys.path
# instead of copying them, so this notebook always uses the one shared implementation.
COURSE_DIR = Path("../../tools_and_agent").resolve()
if str(COURSE_DIR) not in sys.path:
    sys.path.insert(0, str(COURSE_DIR))

from typing import Optional

import wikipedia
from common import configure_wikipedia, get_model, traced
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnableLambda
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pydantic import BaseModel, Field

configure_wikipedia()

In [ ]:
model = get_model()


def structured(schema):
    return model.with_structured_output(schema, method="function_calling")

## Tagging

Ask for a fixed shape - sentiment and language - back for any input text.

In [ ]:
class Tagging(BaseModel):
    """Tag the piece of text with particular info."""
    sentiment: str = Field(description="sentiment of text, should be `pos`, `neg`, or `neutral`")
    language: str = Field(description="language of text (should be ISO 639-1 code)")


tagging_prompt = ChatPromptTemplate.from_messages(
    [
        ("system", "Think carefully, and then tag the text as instructed"),
        ("user", "{input}"),
    ]
)
tagging_chain = tagging_prompt | structured(Tagging)

print(tagging_chain.invoke({"input": "I love langchain"}, config=traced("L4: tagging English")))

In [ ]:
print(tagging_chain.invoke({"input": "non mi piace questo cibo"}, config=traced("L4: tagging Italian")))

## Extraction

Pull a *list* of structured items (people, with an optional age) out of free text - the schema
nests one Pydantic model inside another.

In [ ]:
class Person(BaseModel):
    """Information about a person."""

    name: str = Field(description="person's name")
    age: Optional[int] = Field(default=None, description="person's age")


class Information(BaseModel):
    """Information to extract."""

    people: list[Person] = Field(description="List of info about people")


extraction_prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "Extract the relevant information, if not explicitly provided do not guess. Extract partial info",
        ),
        ("human", "{input}"),
    ]
)
extraction_chain = extraction_prompt | structured(Information)

result = extraction_chain.invoke({"input": "Joe is 30, his mom is Martha"}, config=traced("L4: extraction"))
result.people

## Doing it for real: tag and extract from a live Wikipedia article

Fetch a real article and run the same tagging/extraction machinery over it.

In [ ]:
article_title = "Large language model"
page_content = wikipedia.page(title=article_title, auto_suggest=False).content[:10000]

print(f"fetched {len(page_content)} chars from the '{article_title}' Wikipedia article")
print(page_content[:300])

In [ ]:
class Overview(BaseModel):
    """Overview of a section of text."""

    summary: str = Field(description="Provide a concise summary of the content.")
    language: str = Field(description="Provide the language that the content is written in.")
    keywords: str = Field(description="Provide keywords related to the content.")


overview_prompt = ChatPromptTemplate.from_messages(
    [
        ("system", "Think carefully, and then tag the text as instructed"),
        ("user", "{input}"),
    ]
)
overview_chain = overview_prompt | structured(Overview)

overview_chain.invoke({"input": page_content}, config=traced("L4: overview tagging"))

Extract every paper mentioned in the article, as a structured list.

In [ ]:
class Paper(BaseModel):
    """Information about a paper mentioned in the text."""

    title: str
    author: Optional[str] = None


class Info(BaseModel):
    """Information to extract."""

    papers: list[Paper]


paper_prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "A article will be passed to you. Extract from it all papers that are "
            "mentioned by this article. Do not extract the name of the article itself. "
            "If no papers are mentioned that's fine - you don't need to extract any! "
            "Just return an empty list. Do not make up or guess ANY extra information. "
            "Only extract what exactly is in the text.",
        ),
        ("human", "{input}"),
    ]
)
paper_extraction_chain = paper_prompt | structured(Info) | RunnableLambda(lambda info: info.papers)

paper_extraction_chain.invoke({"input": page_content}, config=traced("L4: paper extraction (single)"))

One call only sees the first 10k characters. To cover the whole article, split it into chunks
and run extraction over every chunk, tolerating the occasional chunk whose output doesn't match
the schema (real text sometimes makes the model emit, say, `author` as a list instead of a
string).

In [ ]:
def flatten(matrix):
    flat_list = []
    for row in matrix:
        flat_list += row
    return flat_list


text_splitter = RecursiveCharacterTextSplitter(chunk_size=2000, chunk_overlap=0)
splits = text_splitter.split_text(page_content)
print(f"split article into {len(splits)} chunks")

prep = RunnableLambda(lambda x: [{"input": doc} for doc in text_splitter.split_text(x)])


def batch_tolerating_bad_chunks(docs, config=None):
    # `.map()` would let one bad chunk crash the whole extraction, so batch with
    # `return_exceptions=True` and drop only the chunks that failed to parse.
    # RunnableLambda injects the parent `config` (declared as a param here), so the
    # Langfuse callback passed to `full_extraction_chain.invoke(...)` below reaches
    # this nested `.batch()` call too.
    results = paper_extraction_chain.batch(docs, config=config, return_exceptions=True)
    return [r for r in results if not isinstance(r, Exception)]


full_extraction_chain = prep | RunnableLambda(batch_tolerating_bad_chunks) | flatten

full_extraction_chain.invoke(page_content, config=traced("L4: paper extraction (chunked)"))